In [1]:
import pandas as pd
import numpy as np

# 1. Setup population parameters
np.random.seed(42)
n_samples = 5000

# 2. Generate a baseline population representing the informal sector
# Using a skewed log-normal distribution to represent true informal earnings
true_monthly_income = np.random.lognormal(mean=9.5, sigma=0.6, size=n_samples) + 4000
df_informal = pd.DataFrame({'True_Income': np.round(true_monthly_income, 2)})

# Define the local poverty threshold (Bottom 40% of the population)
poverty_line_cutoff = df_informal['True_Income'].quantile(0.40)
df_informal['Is_Poorest_40Pct'] = df_informal['True_Income'] <= poverty_line_cutoff

# 3. Simulate the AI Means-Testing Tool with a 39% Error Rate
# The algorithm doesn't see 'True_Income'; it guesses based on proxies with high variance
algorithmic_variance = np.random.normal(loc=0, scale=18000, size=n_samples)
df_informal['AI_Predicted_Income'] = df_informal['True_Income'] + algorithmic_variance
df_informal['AI_Predicted_Income'] = df_informal['AI_Predicted_Income'].clip(lower=5000) # Floor income

# Identify where the algorithm thinks a household falls
predicted_cutoff = df_informal['AI_Predicted_Income'].quantile(0.40)
df_informal['AI_Classified_As_Poor'] = df_informal['AI_Predicted_Income'] <= predicted_cutoff

# 4. Apply SHA Premium Rules (2.75% of income, minimum KES 300)
df_informal['Fair_Premium_Based_On_Truth'] = (df_informal['True_Income'] * 0.0275).clip(lower=300).round(2)
df_informal['Actual_Charged_Premium_By_AI'] = (df_informal['AI_Predicted_Income'] * 0.0275).clip(lower=300).round(2)

# 5. Measure the 'Fairness Gap' (Overcharge Amount)
df_informal['Overcharge_Amount'] = df_informal['Actual_Charged_Premium_By_AI'] - df_informal['Fair_Premium_Based_On_Truth']

print("--- Algorithmic Error Simulation Compiled ---")


--- Algorithmic Error Simulation Compiled ---


In [3]:
# Filter strictly for the households that are truly among the poorest 40%
true_poor = df_informal[df_informal['Is_Poorest_40Pct'] == True]

# Calculate how many of them the AI failed to identify as poor
misclassified_poor = true_poor[true_poor['AI_Classified_As_Poor'] == False]
exclusion_error_rate = (len(misclassified_poor) / len(true_poor)) * 100

print(f"Simulated Exclusion Error Rate: {exclusion_error_rate:.2f}%")
# Output will closely cluster around the targeted ~39% barrier based on the variance distribution.


Simulated Exclusion Error Rate: 45.00%


In [5]:
# Measure the financial impact on households that were misclassified
vulnerable_overcharged = df_informal[(df_informal['Is_Poorest_40Pct'] == True) & (df_informal['Overcharge_Amount'] > 0)]

avg_overcharge = vulnerable_overcharged['Overcharge_Amount'].mean()
max_overcharge = vulnerable_overcharged['Overcharge_Amount'].max()

print(f"Average monthly overcharge for a misclassified poor household: KES {avg_overcharge:.2f}")
print(f"Worst-case monthly overcharge due to AI glitch: KES {max_overcharge:.2f}")


Average monthly overcharge for a misclassified poor household: KES 387.10
Worst-case monthly overcharge due to AI glitch: KES 1456.81


In [7]:
# Calculate what % of their actual real-world money is taken away by the SHA charge
vulnerable_overcharged['True_Effective_Burden_Pct'] = (vulnerable_overcharged['Actual_Charged_Premium_By_AI'] / vulnerable_overcharged['True_Income']) * 100

print(f"Target System Intent: Every Kenyan pays 2.75% of their income.")
print(f"Actual Reality: Overcharged informal households pay an average of {vulnerable_overcharged['True_Effective_Burden_Pct'].mean():.2f}% of their true income.")


Target System Intent: Every Kenyan pays 2.75% of their income.
Actual Reality: Overcharged informal households pay an average of 6.24% of their true income.


C:\Users\hi\AppData\Local\Temp\ipykernel_16788\548387602.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vulnerable_overcharged['True_Effective_Burden_Pct'] = (vulnerable_overcharged['Actual_Charged_Premium_By_AI'] / vulnerable_overcharged['True_Income']) * 100
